# Pulling daily prices with `esg_adaptive_rl.data`

**Goal:** pull daily price history for a given company (ticker) over an arbitrary date range,
and sanity-check what the project's data layer actually hands to the RL environment.

**Why no new fetcher / no Alpha Vantage key:** the capability already lives in the repo.
`ESG_Adaptive_RL/esg_adaptive_rl/data.py` wraps `yfinance.download(..., auto_adjust=True)`
in `load_market_data(tickers, start, end)`. Using it here — rather than writing a parallel
downloader — means this notebook exercises the *exact* code path that training uses, so any
data bug it surfaces is a real bug.

This notebook lives **outside** the `ESG_Adaptive_RL` git repo and does not modify it.

Two things worth knowing up front:

1. `load_market_data` returns **daily simple returns**, not price levels. The adjusted close
   is computed internally and discarded. Cell 5 reconstructs the price path from returns.
2. The `esg` field is a deterministic **synthetic placeholder**, not real ESG data — this is
   documented in `data.py` itself. See cell 8.


## 1. Setup and path bootstrap

In [ ]:
import sys
from pathlib import Path

# The package is a sibling directory, not an installed distribution, so put its
# root on sys.path. Path.cwd() is notebooks/ when run interactively; fall back to
# an explicit search so `jupyter nbconvert --execute` from the repo root also works.
_here = Path.cwd()
_repo = next(
    (p / "ESG_Adaptive_RL" for p in (_here, _here.parent, _here.parent.parent)
     if (p / "ESG_Adaptive_RL" / "esg_adaptive_rl" / "data.py").exists()),
    None,
)
assert _repo is not None, "Could not locate ESG_Adaptive_RL relative to %s" % _here
sys.path.insert(0, str(_repo))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

import esg_adaptive_rl
from esg_adaptive_rl.data import ESG_FACTORS, MarketData, load_market_data, split_by_date
from esg_adaptive_rl import config

print("esg_adaptive_rl", esg_adaptive_rl.__version__, "from", Path(esg_adaptive_rl.__file__).parent)
print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

: 

## 2. Parameters

Everything company- and date-specific is confined to this cell — change `TICKER` / `START` /
`END` and re-run the notebook top to bottom for any other name.

In [ ]:
TICKER = "AAPL"
START = "2000-01-01"   # inclusive
END = "2026-07-01"     # exclusive (yfinance convention)

# Write the pulled series to data/ in the final cell.
SAVE = False

print(f"Requested: {TICKER}  {START} -> {END}")
print()
print("Project defaults from esg_adaptive_rl.config, for reference:")
print(f"  START_DATE = {config.START_DATE}")
print(f"  END_DATE   = {config.END_DATE}")
print(f"  SPLIT_DATE = {config.SPLIT_DATE}")
print(f"  UNIVERSE   = {len(config.UNIVERSE)} names: {', '.join(config.UNIVERSE)}")

## 3. Pull the single ticker

`load_market_data` takes a *list* of tickers, so a single name is a one-element list. It
returns a `MarketData` dataclass whose arrays all share one time axis and one asset axis.

In [ ]:
data = load_market_data([TICKER], START, END)

print(f"tickers : {data.tickers}")
print(f"returns : {data.returns.shape}  (T x N)")
print(f"dates   : {data.dates[0].date()} -> {data.dates[-1].date()}  ({len(data.dates)} trading days)")
print(f"esg     : {{{', '.join(f'{k}: {v.shape}' for k, v in data.esg.items())}}}")

# A tidy frame is easier to inspect than the raw ndarray.
returns_df = pd.DataFrame(data.returns, index=data.dates, columns=data.tickers)
returns_df.head()

## 4. Sanity checks

Cheap assertions that catch the failure modes that actually bite: silent gaps, NaNs leaking
into the return matrix, and unadjusted splits masquerading as a -50% day.

In [ ]:
r = data.returns[:, 0]
span_days = (data.dates[-1] - data.dates[0]).days
expected = span_days / 365.25 * 252  # ~252 US trading days per year

print(f"Calendar span      : {span_days} days ({span_days / 365.25:.2f} years)")
print(f"Trading days found : {len(r)}  (expected ~{expected:.0f}, ratio {len(r) / expected:.3f})")
print(f"NaN / inf          : {np.isnan(r).sum()} / {np.isinf(r).sum()}")
print(f"Daily return  mean : {r.mean():+.5f}   std: {r.std(ddof=1):.5f}")
print(f"              min  : {r.min():+.4f}    max: {r.max():+.4f}")
print(f"Annualised    mean : {r.mean() * 252:+.2%}   vol: {r.std(ddof=1) * np.sqrt(252):.2%}")

assert not np.isnan(data.returns).any(), "NaNs in the return matrix"
assert not np.isinf(data.returns).any(), "infs in the return matrix"
assert data.dates.is_monotonic_increasing and data.dates.is_unique, "date index is not clean"

# |r| > 25% in a single day on a large-cap name usually means an unadjusted corporate action.
suspects = returns_df[returns_df.abs().max(axis=1) > 0.25]
if len(suspects):
    print(f"\n{len(suspects)} day(s) with |return| > 25% — check for unadjusted splits:")
    display(suspects)
else:
    print("\nNo |return| > 25% days. No obvious adjustment errors.")

# Largest calendar gap between consecutive observations (a long holiday is fine; a month is not).
gaps = data.dates.to_series().diff().dropna()
print(f"Largest gap between trading days: {gaps.max().days} days, ending {gaps.idxmax().date()}")

## 5. Reconstruct the price level and cross-check against yfinance

`load_market_data` discards the price level, but it is recoverable: the returns are simple
returns off the adjusted close, so `(1 + r).cumprod()` rebuilds the path up to the (unknown)
starting price. Rebasing the independently-downloaded adjusted close the same way makes the
two directly comparable, which verifies the module's internal price handling.

In [ ]:
# Reconstructed from the module's returns, rebased to 100 at the first return date.
reconstructed = 100.0 * (1.0 + returns_df[TICKER]).cumprod()

# Independent pull of the same ticker/range, straight from yfinance.
raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
raw_close = raw["Close"]
if isinstance(raw_close, pd.DataFrame):   # (field, ticker) MultiIndex columns
    raw_close = raw_close[TICKER]
raw_close = raw_close.dropna()

# Rebase yfinance's close onto the same anchor: the last price *before* the first return.
anchor = raw_close.loc[: reconstructed.index[0]].iloc[-2]
rebased = 100.0 * raw_close.reindex(reconstructed.index) / anchor

prices = pd.DataFrame({
    "adj_close": raw_close.reindex(reconstructed.index),
    "reconstructed_100": reconstructed,
    "yfinance_100": rebased,
    "daily_return": returns_df[TICKER],
})

deviation = (prices["reconstructed_100"] / prices["yfinance_100"] - 1.0).abs().max()
print(f"Max abs relative deviation, reconstructed vs. yfinance: {deviation:.3e}")
assert deviation < 1e-8, "Reconstructed price path diverges from the raw adjusted close"
print("Price handling inside load_market_data verified.\n")

prices.head()

## 6. Plot the price path and daily returns

In [ ]:
fig, (ax_price, ax_ret) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)

ax_price.plot(prices.index, prices["adj_close"], lw=1.2, color="#1f4e79")
ax_price.set_title(f"{TICKER} — split/dividend-adjusted close, {START} to {END}")
ax_price.set_ylabel("Adjusted close (USD)")
ax_price.grid(alpha=0.3)

ax_ret.plot(prices.index, prices["daily_return"], lw=0.6, color="#7f7f7f")
ax_ret.axhline(0.0, color="black", lw=0.8)
ax_ret.set_ylabel("Daily simple return")
ax_ret.set_xlabel("Date")
ax_ret.grid(alpha=0.3)

# Annualised realised vol on a 21-day rolling window, for context on the return panel.
ax_vol = ax_ret.twinx()
rolling_vol = prices["daily_return"].rolling(21).std() * np.sqrt(252)
ax_vol.plot(prices.index, rolling_vol, lw=1.0, color="#c00000", alpha=0.8)
ax_vol.set_ylabel("21d realised vol (ann.)", color="#c00000")
ax_vol.tick_params(axis="y", labelcolor="#c00000")

fig.tight_layout()
plt.show()

## 7. Multi-ticker pull

The same call with several tickers shows the two behaviours that matter for portfolio work:
columns come back in the **requested order** (not yfinance's), and `data.py` does an inner
join then `dropna(how="any")` — so the panel is truncated to days on which *every* name
traded. That is what keeps the return matrix rectangular for the environment, but it also
means one short-history name silently shortens the whole sample.

In [ ]:
sp500_2000 = pd.read_csv('../ESG_Adaptive_RL/Dataset/tickers_sp500_2000.csv')
sp500_2000

In [ ]:
sp500_now = pd.read_csv('../ESG_Adaptive_RL/Dataset/tickers_sp500_now.csv')

### 7a. Audit the universe before pulling it

Intersecting the Jan-2000 list with today's list gives *candidates*, not a usable universe.
Two things must be checked first, and they are different problems:

**Price continuity.** `data.py` ends with `dropna(how="any")`, so the panel is truncated to
its latest-starting column. A single ticker whose history begins in 2022 costs you 22 years
on all 190 others. This is the automated drop rule.

**Identifier collision.** A ticker string is not an identifier — symbols get recycled. `CEG`
was Constellation Energy Group (absorbed by Exelon in 2012) and is now Constellation Energy
Corporation (spun *out* of Exelon in 2022). Matching on the string silently equates them.

`universe_audit.py` documents why price continuity is the only test safe to automate, and
why the SEC-CIK signal is surfaced as evidence for review rather than used as a filter.
The short version: CIK flags Disney, Oracle and Exxon as "new entities" because of holdco
reorganisations, and fuzzy name matching scores `Constellation Energy Corp` at 1.00 against
`Constellation Energy Group` — confidently wrong on the one case it most needs to catch.

In [ ]:
from universe_audit import audit_universe

candidates = sorted(set(sp500_now["tickers"]) & set(sp500_2000["tickers"]))
print(f"{len(sp500_2000)} names in Jan 2000, {len(sp500_now)} today "
      f"-> {len(candidates)} shared ticker strings\n")

# Cold cache issues one SEC request per candidate (rate-limited); afterwards it is local.
audit = audit_universe(candidates, start=START, end=END)
audit.to_csv("../reference/universe_audit.csv", index=False)

In [ ]:
dropped = audit.loc[~audit["keep"]]
review = audit.loc[audit["keep"] & audit["review"]]

print(f"DROPPED — no price history at {START} ({len(dropped)} of {len(audit)}):")
display(dropped[["ticker", "sec_name", "first_price", "missing_years"]]
        .style.format({"first_price": lambda d: d.date().isoformat(),
                       "missing_years": "{:.1f}"}))

print(f"\nFLAGGED FOR REVIEW — registrant postdates {START}, but history is continuous "
      f"({len(review)}). These are kept: the CIK is usually new because of a holdco "
      f"reorganisation or redomiciliation, not a ticker collision. Spot-check before "
      f"publishing any result that leans on them.")
display(review[["ticker", "sec_name", "cik", "first_filing"]]
        .style.format({"first_filing": lambda d: d.date().isoformat()}))

# sorted(), not list(set(...)): set iteration order over strings is randomised per
# process, which would make the column order - and anything seeded off it - vary run to run.
BASKET = sorted(audit.loc[audit["keep"], "ticker"])
print(f"\nFinal universe: {len(BASKET)} tickers")

In [ ]:
basket = load_market_data(BASKET, START, END)
basket_df = pd.DataFrame(basket.returns, index=basket.dates, columns=basket.tickers)

print(f"Column order preserved: {basket.tickers == BASKET}")
print(f"Panel shape: {basket.returns.shape}")
print(f"Dates: {basket.dates[0].date()} -> {basket.dates[-1].date()}")

# The audit should have removed every binding constraint, so the panel now spans the
# full requested window rather than being clipped to the latest-listing name.
span_years = (basket.dates[-1] - basket.dates[0]).days / 365.25
print(f"Span: {span_years:.1f} years of {(pd.Timestamp(END) - pd.Timestamp(START)).days / 365.25:.1f} requested")

summary = pd.DataFrame({
    "ann_return": basket_df.mean() * 252,
    "ann_vol": basket_df.std(ddof=1) * np.sqrt(252),
})
summary["sharpe_0rf"] = summary["ann_return"] / summary["ann_vol"]
display(summary.sort_values("sharpe_0rf", ascending=False).head(15).style.format("{:.3f}"))

print("Daily-return correlation (first 15 names):")
display(basket_df.iloc[:, :15].corr().style.format("{:.2f}")
        .background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

In [ ]:
basket_df

In [ ]:
basket_df.to_csv("../ESG_Adaptive_RL/Dataset/daily_returns.csv", index_label="date")

> **Standing caveat for the write-up.** This universe is every company that was in the index
> in Jan 2000 *and* is in it today — survivors only. Robot Wealth measures the resulting
> upward bias at roughly **double** the true return, so absolute performance from this panel
> is not reportable. Relative comparisons across reward weightings remain meaningful, since
> every configuration is handicapped identically. Two further distortions ride on top: the
> re-IPO'd names dropped above (HCA, DG, HLT, DELL) bias *which* survivors are excluded, and
> recycled tickers admit companies that were never in the 2000 index at all.

## 8. The ESG table is a placeholder — do not read anything into it

`data.py` builds `esg` from a per-ticker MD5 seed plus a sinusoidal drift. It is deterministic
and reproducible, and it enters the environment as a *time-indexed per-asset* array so that a
real ESG history can be dropped in with no change to the environment or the agent. But the
**values carry no information about the company**.

Real ESG histories appear to be sitting unused at `ESG_Adaptive_RL/Dataset/` (`ESG_2000-2026.csv`,
`ESG_SP500.xlsx`, `ESG_last10yrs.xlsx`). Wiring those in — point-in-time, with the rating's
*publication* date rather than its as-of date, to avoid look-ahead — is the natural next step.

In [ ]:
esg_df = pd.DataFrame(
    {factor: data.esg[factor][:, 0] for factor in ESG_FACTORS},
    index=data.dates,
)

print(f"Synthetic E/S/G for {TICKER} — range and variation:")
display(esg_df.describe().T[["mean", "std", "min", "max"]].style.format("{:.4f}"))

fig, ax = plt.subplots(figsize=(12, 3.2))
for factor in ESG_FACTORS:
    ax.plot(esg_df.index, esg_df[factor], lw=1.0, label=factor)
ax.set_title(f"{TICKER} — SYNTHETIC placeholder E/S/G (not real data)")
ax.set_ylabel("Score in [0, 1]")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", ncol=3)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# Same seed -> same table. This is the reproducibility guarantee _stable_seed() buys.
again = load_market_data([TICKER], START, END)
assert np.allclose(again.esg["E"], data.esg["E"]), "ESG placeholder is not reproducible"
print("Placeholder ESG reproduces exactly across calls.")

## 9. Chronological train/test split

`split_by_date` cuts on a date rather than a row fraction, keeping the test period strictly in
the future relative to training. This is the single most important guard against look-ahead in
a backtest, so it is worth asserting rather than assuming.

In [ ]:
train, test = split_by_date(data, config.SPLIT_DATE)

for name, split in (("train", train), ("test", test)):
    if len(split.dates):
        print(f"{name:>5}: {len(split.dates):>5} days   "
              f"{split.dates[0].date()} -> {split.dates[-1].date()}   "
              f"returns {split.returns.shape}")
    else:
        print(f"{name:>5}: empty (SPLIT_DATE {config.SPLIT_DATE} lies outside the pulled range)")

assert train.dates.max() < test.dates.min(), "train and test overlap in time"
assert len(train.dates) + len(test.dates) == len(data.dates), "split lost or duplicated rows"
assert pd.Timestamp(config.SPLIT_DATE) > train.dates.max(), "train leaks past the split date"
print("\nNo temporal overlap; the split is exhaustive and leak-free.")

## 10. Optional export

Set `SAVE = True` in cell 2 to persist. `data/` is gitignored at the repo root.

In [ ]:
if SAVE:
    out_dir = _repo.parent / "data"
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"{TICKER}_{START}_{END}.csv"
    prices.to_csv(out_path, index_label="date")
    print(f"Wrote {len(prices)} rows -> {out_path}")
else:
    print("SAVE is False — nothing written. Set SAVE = True in cell 2 to export.")
    display(prices.tail())